In [ ]:
# DEFAULT
import importlib
import os
import requests
import time
#3RD PARTY
import pandas as pd
#CUSTOM
import Classes.aggregator as agg
import Classes.evaluator as ev
from utils import pick_pictures_randomly as ppr

<module 'Classes.evaluator' from 'c:\\Users\\mernst\\Nextcloud\\Voucher classifier\\master_thesis\\Classes\\evaluator.py'>

In [ ]:
# Load data for random sampling
data = pd.read_csv('data/herbonauts_data/herbonauts_data_20250901.csv', sep=';') # All collected Herbonauten records and associated metadata
data = data[data['Barcode'].str.startswith('B 1')] # Only include "regular" specimens, indicated by the first number in the catalog number
data.to_csv('data/herbonauts_data/herbonauts_data_existing_images_20250901_filtered.csv', sep=';', index=False) # store them back as filtered version

In [ ]:
# Pick randomly around 1800 specimen records with stratification by family for the sake of representativeness compared to Dillen et al. 2019
importlib.reload(ppr)
data_path = ppr.pick_random_catalog_numbers_stratified_families('data/herbonauts_data/herbonauts_data_existing_images_20250901_filtered.csv', 1800, 'Barcode')

C:\Users\mernst\Nextcloud\Voucher classifier\master_thesis\utils\pick_pictures_randomly.py:34: DtypeWarning: Columns (8,9,12,13,20,21,24,31,32,42,43) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, sep=';')


Length: 1838
Stratified sampled catalog numbers saved to data/stratified_sample_catalog_numbers_20251215_132626.csv


In [ ]:
# Filter the Herbonauten dataset for the selected catalog numbers in the random sample
print(data_path)
barcodes = pd.read_csv(data_path)['Barcode'].tolist()
data = pd.read_csv('data/herbonauts_data/herbonauts_data_20250901.csv', sep=';')
data = data[data['Barcode'].isin(barcodes)]
selected_catalog_numbers_path = 'data/herbonauts_data/' + data_path.split('/')[-1].replace('.csv', '_selected.csv')
data.to_csv(selected_catalog_numbers_path, index=False)

data/stratified_sample_catalog_numbers_20251215_132626.csv


C:\Users\mernst\AppData\Local\Temp\ipykernel_27996\57549487.py:4: DtypeWarning: Columns (8,9,12,13,20,21,24,31,32,42,43) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('data/herbonauts_data/herbonauts_data_20250901.csv', sep=';')


In [ ]:
# Read in the selected ground truth data
data = pd.read_csv(selected_catalog_numbers_path)
len(data)

1838

In [ ]:
def download_images(csv_path, no_of_images, output_dir='data/sample_images_stratified_after_filter/'):
    """
    This function downloads images based on the selected catalog numbers from the stratified Herbonauten sample. 
    For this, it reads the CSV, which contains one column 'Barcode' with catalogue numbers that can be used to construct image URLs.
    The downloaded images are then saved in the data/sample_images_stratified/ directory or any other specified directory.

    Parameters
    ----------
    csv_path: str
        Path of the CSV that includes a random (stratified) sample
    no_of_images: int
        Number of samples in the respective CSV
    output_dir: str, Optional
        Output folder
    """
    df = pd.read_csv(csv_path)
    if not no_of_images or no_of_images > len(df):
        no_of_images = len(df)
    start_time = time.time()
    failed_downloads = [] # Memory for specimen images that could not be downloaded

    for index, row in df.sample(n=no_of_images).iterrows(): # .sample as legacy code for testing
        if index % 100 == 0: # print state after every 100 images
            percent_complete = (index / no_of_images) * 100
            print(f'Progress: {percent_complete:.2f}% ({index}/{no_of_images}): {time.time() - start_time:.2f} seconds elapsed')
        catalog_number = row['Barcode']
        catalog_number = str(catalog_number).replace(' ', '')
        image_url = "https://image.bgbm.org/images/internal/HerbarThumbs/" + catalog_number + "_1700"
        image_path = "./" + output_dir + "/" + catalog_number + ".jpg"

        if not os.path.exists(image_path):
            try:
                response = requests.get(image_url, stream=True)
                if response.status_code == 200:
                    with open(image_path, 'wb') as file:
                        for chunk in response.iter_content(1024):
                            file.write(chunk)
                else:
                    failed_downloads.append(catalog_number)
                    None
            except Exception as e:
                failed_downloads.append(catalog_number)
                print("Error downloading" + image_url + ":" + e)
        else:
            print("Image " + image_path + " already exists. Skip download.")
    end_time = time.time()
    print("Time needed for collecting images: " + str(end_time - start_time) + " seconds")

    return failed_downloads
failed_downloads = download_images('data/herbonauts_data/stratified_sample_catalog_numbers_20251117_103411_selected.csv', len(data))

Progress: 21.76% (400/1838): 5.14 seconds elapsed
Progress: 76.17% (1400/1838): 44.57 seconds elapsed
Progress: 54.41% (1000/1838): 102.00 seconds elapsed
Progress: 10.88% (200/1838): 107.90 seconds elapsed
Progress: 38.08% (700/1838): 119.23 seconds elapsed
Error downloading https://image.bgbm.org/images/internal/HerbarThumbs/B101072758_1700: HTTPSConnectionPool(host='image.bgbm.org', port=443): Max retries exceeded with url: /images/internal/HerbarThumbs/B101072758_1700 (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x000001DBDF64DE50>, 'Connection to image.bgbm.org timed out. (connect timeout=None)'))
Progress: 70.73% (1300/1838): 245.47 seconds elapsed
Progress: 48.97% (900/1838): 267.47 seconds elapsed
Progress: 0.00% (0/1838): 287.92 seconds elapsed
Progress: 81.61% (1500/1838): 344.61 seconds elapsed
Progress: 92.49% (1700/1838): 351.66 seconds elapsed
Progress: 32.64% (600/1838): 355.95 seconds elapsed
Progress: 65.29% (1200/1838): 436.39 seconds e

Transcription needs to take place here. Please switch to /notebooks/transcription/notebook_transcription.ipynb, execute and come back to proceed with training data creation

In [ ]:
# Aggregation of transcription results
importlib.reload(agg)
aggregator = agg.Aggregator(['hespi', 'vouchervision'])
# aggregator.add_transcriptions('hespi', 'data/transcriptions/hespi-results.csv') # uncomment if hespi should be included
aggregator.add_transcriptions('vouchervision', 'data/transcriptions/vv-results.xlsx')
aggregator.aggregate()
res = aggregator.get_aggregated_data()
aggregator.save_aggregated_data('data/transcriptions/aggregated_transcriptions.csv')

Aggregating data from tool: hespi with 0 records.
Aggregating data from tool: vouchervision with 1834 records.


In [43]:
res

,specificEpithet,infraspecificEpithet,genus,scientificNameAuthorship,family,recordedBy,day,month,year,locality,...,scientificName,tool_name,associatedCollectors,eventDateEnd,habitat,minimumElevationInMeters,maximumElevationInMeters,county,stateProvince,specimenDescription
0,macrostachyum,,Gnetum,Hook.,,,,,,,...,Gnetum macrostachyum,hespi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,microcarpum,,Fretum,Bl.,,,,,,,...,Fretum microcarpum,hespi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,,,,,,Praet,,fract,,#,...,NaN,hespi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ferruginea,,Rapanea,(RAP) Mez,Myrsinaceae,B. Rambo SJ,18,7,19490,"RGS, Pareci p. Montenegro",...,Rapanea ferruginea,hespi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,buchanani,,Stemodiopsis,Skan,,S. Paulo,,,1962,Tanganyika :,...,Stemodiopsis buchanani,hespi,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3445,obcordatus,NaN,Dianthus,Marg. Reut,NaN,"Dörfler,I.",<NA>,<NA>,1893,Macedon. centr.,...,Dianthus obcordatus,vouchervision,,<NA>,In pratis ad Rošzdan.,,,,,
3446,tubulosa,NaN,Corylus,W.,NaN,"Dörfler,Ignatii",4,8,1887,Fundort in der Nähe des Grünbergerguthes aufde...,...,Corylus tubulosa,vouchervision,,<NA>,,,,,Ober Österreich,"Gmunden, ein grosser Strauch unter C. Avell."
3447,betulus,NaN,Carpinus,L.,NaN,"Bornmüller,J. J.",<NA>,9,1932,Woimar Wald bri Liigafold,...,Carpinus betulus L. f. fol. duplicato-lacoro- ...,vouchervision,,<NA>,,,,,,
3448,carthusianorum,NaN,Dianthus,,NaN,"Dörfler,J.",NaN,NaN,NaN,Ad montem Rareu pr.. Kimpolung in Bukowina.,...,Dianthus carthusianorum,vouchervision,,<NA>,,,,,Bukowina,


In [ ]:
# Evaluation of aggregated transcription results
importlib.reload(ev)

GROUND_TRUTH_RENAME_DICT = {
    'Barcode': 'catalogNumber',
    'Family': 'family',
    'taxon': 'scientificName',
    'genus': 'genus',
    'spepi': 'specificEpithet',
    'inepi': 'infraspecificEpithet',
    'country': 'country',
    'collect_date_begin': 'eventDate',
    'collect_date_end': 'eventDateEnd',
    'collector': 'recordedBy',
    'collect_number - ': 'collector_number_ocr_results',
    'locality': 'locality',
    'geo - position': 'geolocation_ocr_results',
    'geo - radius': 'geolocation_ocr_results',
    'geo - description': 'geolocation_ocr_results'
}

evaluator = ev.Evaluator()
df = pd.read_csv('data/herbonauts_data/stratified_sample_catalog_numbers_20251117_103411_selected.csv')
df = df.rename(columns=GROUND_TRUTH_RENAME_DICT)
evaluator.load_ground_truth(df)
gt = evaluator.ground_truth_data
evaluator.add_transcription_results(res)
evaluation_results = evaluator.evaluate_all()

Loaded 1838 ground truth specimen records.
Loaded 1834 transcription results from DataFrame.
Successfully evaluated 1834 transcriptions.


In [ ]:
# Export transcription evaluation as a CSV file
evaluator.export_results(evaluation_results, 'data/evaluation/aggregated_transcription_evaluation.csv')
res_for_export = evaluator.results_for_export
res_for_export

Exporting results to 'data/evaluation/aggregated_transcription_evaluation.csv'.
Folder exists: True
Creating folder 'data/evaluation' for export.
Export to 'data/evaluation/aggregated_transcription_evaluation.csv' successful.


[{'catalogNumber': 'B100001493',
  'tool_name': 'vouchervision',
  'composite_score': 0.27796610169491526,
  'quality': 'low',
  'recommended_pipeline': 'herbonauten',
  'taxon_transcription': 'Justica mississipiensis',
  'taxon_ground_truth': 'Gnetum microcarpum',
  'taxon_exact_match': False,
  'taxon_composite_score': 0.38983050847457623,
  'taxon_errors': 'Low similarity score; Failed taxonomic validation',
  'collector_transcription': '',
  'collector_ground_truth': 'Praetorius',
  'collector_exact_match': False,
  'collector_composite_score': 0.0,
  'collector_errors': "Variant with highest similarity score: '' with score 0.00; Low collector name similarity: 0.00",
  'collection_date_transcription': '<NA>',
  'collection_date_ground_truth': 'nan',
  'collection_date_exact_match': True,
  'collection_date_composite_score': 1.0,
  'collection_date_errors': '',
  'locality_transcription': '',
  'locality_ground_truth': 'Sumatra',
  'locality_exact_match': False,
  'locality_composit

In [ ]:
# collect catalog numbers and label them for training dataset with the tool name that produces the best results
training_dataset_entries = []
if 'evaluator' in locals():
    transcription_results = evaluator.results_for_export
else:
    transcription_results = pd.read_csv('data/evaluation/aggregated_transcription_evaluation.csv').to_dict(orient='records')
transcription_results_df = pd.DataFrame(transcription_results)
# for the thesis data denoising, manual_review records are excluded. uncomment if they should be in the training dataset
transcription_results_df = transcription_results_df[transcription_results_df['recommended_pipeline'] != 'manual review']
# grouping  to find the best tool for groups of catalog numbers (only important if more than one transcription tool have been evaluated)
for catalog_number, group in transcription_results_df.groupby('catalogNumber'): 
    max_score = 0.0
    max_pipeline_name = ""
    for index, row in group.iterrows():
        if row['composite_score'] > max_score:
            max_score = row['composite_score']
            max_pipeline_name = row['recommended_pipeline']
    if max_pipeline_name == "manual review":
        max_pipeline_name = "herbonauten"
    training_dataset_entries.append({
        'catalogNumber': catalog_number,
        'best_tool': max_pipeline_name
    })
training_dataset_df = pd.DataFrame(training_dataset_entries)

# training_dataset_df.to_csv('data/training/training_dataset_labels.csv', index=False) 
training_dataset_df.to_csv('data/training/training_dataset_labels_wo_manual_review.csv', index=False)